# OS NGD Buildings – SPARQL Query Tests (rdflib)

This notebook loads the IES Building ontology and the sample fused triple set `sample-ia-node-with-os-ngd.ttl`, then runs the new iris-api NGD queries against an in-memory `rdflib.Graph` — simulating queries over a Fuseki dataset containing all triples.

## Setup

In [1]:
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, RDFS, XSD
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path.cwd()
ONTOLOGY_PATH = NOTEBOOK_DIR.parent.parent / 'materialised-view-creation' / 'src' / 'create-view' / 'ies-building.ttl'
SAMPLE_DATA_PATH = NOTEBOOK_DIR / 'sample-ia-node-with-os-ngd.ttl'

print('Ontology path:', ONTOLOGY_PATH)
print('Sample data path:', SAMPLE_DATA_PATH)

# Namespaces used in queries
IES = Namespace('http://informationexchangestandard.org/ont/ies#')
BUILDING = Namespace('http://ies.data.gov.uk/ontology/ies-building1#')
DATA = Namespace('http://ndtp.co.uk/data#')
QUDT = Namespace('http://qudt.org/schema/qudt/')
UNIT = Namespace('http://qudt.org/vocab/unit/')
QK = Namespace('http://qudt.org/vocab/quantitykind/')

# Load graph (ontology + sample data)
g = Graph()
g.bind('rdf', RDF)
g.bind('rdfs', RDFS)
g.bind('xsd', XSD)
g.bind('ies', IES)
g.bind('building', BUILDING)
g.bind('data', DATA)
g.bind('qudt', QUDT)
g.bind('unit', UNIT)
g.bind('quantitykind', QK)

# Parse ontology first so class hierarchy and properties are present
g.parse(ONTOLOGY_PATH.as_posix(), format='turtle')
# Parse sample data containing both EPC and NGD-derived triples
g.parse(SAMPLE_DATA_PATH.as_posix(), format='turtle')

print(f'Total triples loaded: {len(g):,}')

Ontology path: /home/vagrant/Documents/IRIS-data-pipeline/materialised-view-creation/src/create-view/ies-building.ttl
Sample data path: /home/vagrant/Documents/IRIS-data-pipeline/ngd-property-pipeline/notebooks/sample-ia-node-with-os-ngd.ttl
Total triples loaded: 1,814


## NGD Queries (as in iris-api)
These SPARQL queries mirror the new OS NGD Buildings query helpers in `iris-api/api/query.py`.

In [2]:
def get_ngd_roof_material_for_building(uprn: str) -> str:
    return f"""
        PREFIX building:  <http://ies.data.gov.uk/ontology/ies-building1#>
        PREFIX data:      <http://ndtp.co.uk/data#>
        PREFIX ies:       <http://informationexchangestandard.org/ont/ies#>

        SELECT ?roofMaterial
        WHERE {{
            data:StructureUnit_{uprn} ies:isPartOf ?building .
            ?roof ies:isPartOf ?building .
            ?roofState a building:RoofState ;
                      ies:isStateOf ?roof ;
                      building:isMadeOf ?roofMaterial .
        }}
        LIMIT 1
    """

def get_ngd_solar_panel_presence_for_building(uprn: str) -> str:
    return f"""
        PREFIX building:  <http://ies.data.gov.uk/ontology/ies-building1#>
        PREFIX data:      <http://ndtp.co.uk/data#>
        PREFIX ies:       <http://informationexchangestandard.org/ont/ies#>

        SELECT ?solarPanelPresence
        WHERE {{
            data:StructureUnit_{uprn} ies:isPartOf ?building .
            ?state ies:isStateOf ?building ;
                   a ?solarPanelPresence .
            VALUES ?solarPanelPresence {{
                building:NoSolarPanels
                building:HasSolarPanels
                building:UnknownSolarPanelPresence
            }}
        }}
        LIMIT 1
    """

def get_ngd_roof_shape_for_building(uprn: str) -> str:
    return f"""
        PREFIX building:  <http://ies.data.gov.uk/ontology/ies-building1#>
        PREFIX data:      <http://ndtp.co.uk/data#>
        PREFIX ies:       <http://informationexchangestandard.org/ont/ies#>

        SELECT DISTINCT ?roofShape
        WHERE {{
            data:StructureUnit_{uprn} ies:isPartOf ?building .
            ?shapeState ies:isStateOf ?building ;
                        a building:RoofState ;
                        a ?roofShape .
            VALUES ?roofShape {{
                building:PitchedRoofShape
                building:FlatRoofShape
                building:MixedRoofShape
                building:UnknownRoofShape
            }}
        }}
        LIMIT 1
    """

def get_ngd_roof_aspect_areas_for_building(uprn: str) -> str:
    return f"""
        PREFIX building:     <http://ies.data.gov.uk/ontology/ies-building1#>
        PREFIX data:         <http://ndtp.co.uk/data#>
        PREFIX ies:          <http://informationexchangestandard.org/ont/ies#>
        PREFIX qudt:         <http://qudt.org/schema/qudt/>
        PREFIX unit:         <http://qudt.org/vocab/unit/>
        PREFIX quantitykind: <http://qudt.org/vocab/quantitykind/>

        SELECT ?direction ?m2
        WHERE {{
            data:StructureUnit_{uprn} ies:isPartOf ?building .
            ?roof ies:isPartOf ?building .
            ?roofState a building:RoofState ; ies:isStateOf ?roof .

            ?aspect a ?directionClass ;
                    ies:isPartOf ?roofState ;
                    building:hasCombinedSurfaceArea [
                        building:hasQuantity [
                            qudt:hasQuantityKind quantitykind:Area ;
                            qudt:unit unit:M2 ;
                            qudt:value ?m2
                        ]
                    ] .

            VALUES ?directionClass {{
                building:NorthFacingRoofSectionSum
                building:NorthEastFacingRoofSectionSum
                building:EastFacingRoofSectionSum
                building:SouthEastFacingRoofSectionSum
                building:SouthFacingRoofSectionSum
                building:SouthWestFacingRoofSectionSum
                building:WestFacingRoofSectionSum
                building:NorthWestFacingRoofSectionSum
                building:AreaIndeterminableRoofSectionSum
            }}
            BIND(STRAFTER(STR(?directionClass), '#') AS ?direction)
        }}
    """


## EPC/Building Queries (as in iris-api)
These queries retrieve EPC-derived attributes and general building state potentially impacted by the presence of NGD triples in the same graph.

In [3]:
def get_building(uprn: str) -> str:
    return f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX ies: <http://informationexchangestandard.org/ont/ies#>
        PREFIX building: <http://ies.data.gov.uk/ontology/ies-building1#>
        PREFIX data: <http://ndtp.co.uk/data#>

        SELECT ?lodgementDate ?builtForm ?structureUnitType WHERE {{
            ?structureUnit ies:isIdentifiedBy data:UPRN_{uprn} .
            ?structureUnit a building:StructureUnit .
            ?structureUnitState a building:StructureUnitState .
            ?structureUnitState ies:isStateOf ?structureUnit .

            ?epc_result building:lodgementDate ?lodgementDate .
            ?epc_result ies:isParticipantIn ?epc_assessment .
            ?epc_assessment building:assessedStateForEnergyPerformance ?structureUnitState .

            ?_bf a ?builtForm .
            ?builtForm a building:BuiltForm .
            ?_bf ies:isStateOf ?structureUnit .

            ?_sut a ?structureUnitType .
            ?structureUnitType a building:StructureUnitType .
            ?_sut ies:isStateOf ?structureUnit .
        }}
        LIMIT 1
    """

def get_roof_for_building(uprn: str) -> str:
    return f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX ies: <http://informationexchangestandard.org/ont/ies#>
        PREFIX building: <http://ies.data.gov.uk/ontology/ies-building1#>
        PREFIX data: <http://ndtp.co.uk/data#>

        SELECT ?uprn ?roofConstruction ?roofInsulation ?roofInsulationThickness 
        WHERE {{
            ?structureUnit ies:isIdentifiedBy data:UPRN_{uprn} .
            ?structureUnit a building:StructureUnit .
            ?structureUnitState a building:StructureUnitState .
            ?structureUnitState ies:isStateOf ?structureUnit .

            ?_rc a ?roofConstruction .
            ?roofConstruction a building:RoofConstruction .
            ?_rc ies:isPartOf ?structureUnitState .

            ?_ri a ?roofInsulation .
            ?roofInsulation a building:RoofInsulationLocation .
            ?_ri ies:isPartOf ?structureUnitState .

            OPTIONAL {{
                ?_rit a ?roofInsulationThickness .
                ?roofInsulationThickness a building:RoofInsulationThickness .
                ?_rit ies:isPartOf ?structureUnitState .
            }}
        }}
        LIMIT 1
    """

def get_floor_for_building(uprn: str) -> str:
    return f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX ies: <http://informationexchangestandard.org/ont/ies#>
        PREFIX building: <http://ies.data.gov.uk/ontology/ies-building1#>
        PREFIX data: <http://ndtp.co.uk/data#>

        SELECT ?uprn ?floorConstruction ?floorInsulation WHERE {{

        ?structureUnit ies:isIdentifiedBy data:UPRN_{uprn} .
        ?structureUnit a building:StructureUnit .
        ?structureUnitState a building:StructureUnitState .
        ?structureUnitState ies:isStateOf ?structureUnit .

        ?_fc a ?floorConstruction .
        ?floorConstruction a building:FloorConstruction .
        ?_fc ies:isPartOf ?structureUnitState .

        ?_fi a ?floorInsulation .
        ?floorInsulation a building:FloorInsulation .
        ?_fi ies:isPartOf ?structureUnitState .

        }}
        LIMIT 1
    """

def get_walls_and_windows_for_building(uprn: str) -> str:
    return f"""
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX ies: <http://informationexchangestandard.org/ont/ies#>
        PREFIX building: <http://ies.data.gov.uk/ontology/ies-building1#>
        PREFIX data: <http://ndtp.co.uk/data#>

        SELECT ?uprn ?wallConstruction ?wallInsulation ?windowGlazing WHERE {{

            ?structureUnit ies:isIdentifiedBy data:UPRN_{uprn} .
            ?structureUnit a building:StructureUnit .
            ?structureUnitState a building:StructureUnitState .
            ?structureUnitState ies:isStateOf ?structureUnit .

            ?_wc a ?wallConstruction .
            ?wallConstruction a building:WallConstruction .
            ?_wc ies:isPartOf ?structureUnitState .

            ?_wi a ?wallInsulation .
            ?wallInsulation a building:WallInsulation .
            ?_wi ies:isPartOf ?structureUnitState .

            ?_wg a ?windowGlazing .
            ?windowGlazing a building:GlazingType .
            ?_wg ies:isPartOf ?structureUnitState .

        }}
        LIMIT 1
    """

# def get_fueltype_for_building(uprn: str) -> str:
#     return f"""
#         PREFIX ies:      <http://informationexchangestandard.org/ont/ies#>
#         PREFIX building: <http://ies.data.gov.uk/ontology/ies-building1#>
#         PREFIX data:     <http://ndtp.co.uk/data#>

#         SELECT ?fuelType
#             WHERE {{
#             ?structureUnit ies:isIdentifiedBy data:UPRN_{uprn} .
#             ?structureUnitState ies:isStateOf ?structureUnit .

#             GRAPH <http://ndtp.com/graph/heating-v1> {{
#                 ?structureUnitState building:isServicedBy ?heatingSystem .
#                 ?heatingSystem building:isOperableWithFuel ?fuelType .
#             }}
#         }}
#     """


## Query helpers

In [4]:
def run(query: str):
    """Run a SPARQL query against the in-memory graph and return list of dicts.
    
    Note: rdflib SPARQL returns an rdflib.query.Result with row tuples.
    We'll project to a list of dicts keyed by variable names for convenience.
    """
    res = g.query(query)
    cols = res.vars
    out = []
    for row in res:
        out.append({str(cols[i]): row[i] for i in range(len(cols))})
    return out

def pprint_rows(rows):
    if not rows:
        print('(no results)')
        return
    for i, r in enumerate(rows, 1):
        print(f'Row {i}:')
        for k, v in r.items():
            print(f'  {k}: {v}')
        print()


## Run NGD queries for sample UPRNs
The sample dataset contains links from these UPRNs to NGD-derived building states: `10003320518`, `10067623656`.

In [5]:
uprns = ['10003320518', '10067623656', '10094231108']

for u in uprns:
    print('='*80)
    print('UPRN:', u)
    print('- Roof material')
    rows = run(get_ngd_roof_material_for_building(u))
    pprint_rows(rows)

    print('- Solar panel presence')
    rows = run(get_ngd_solar_panel_presence_for_building(u))
    pprint_rows(rows)

    print('- Roof shape')
    rows = run(get_ngd_roof_shape_for_building(u))
    pprint_rows(rows)

    print('- Roof aspect areas (m2)')
    rows = run(get_ngd_roof_aspect_areas_for_building(u))
    pprint_rows(rows)


UPRN: 10003320518
- Roof material
Row 1:
  roofMaterial: http://ies.data.gov.uk/ontology/ies-building1#TileOrStoneOrSlate

- Solar panel presence
Row 1:
  solarPanelPresence: http://ies.data.gov.uk/ontology/ies-building1#HasSolarPanels

- Roof shape
Row 1:
  roofShape: http://ies.data.gov.uk/ontology/ies-building1#PitchedRoofShape

- Roof aspect areas (m2)
Row 1:
  direction: EastFacingRoofSectionSum
  m2: 0

Row 2:
  direction: NorthFacingRoofSectionSum
  m2: 0

Row 3:
  direction: NorthEastFacingRoofSectionSum
  m2: 0

Row 4:
  direction: NorthWestFacingRoofSectionSum
  m2: 67.1

Row 5:
  direction: SouthFacingRoofSectionSum
  m2: 0

Row 6:
  direction: SouthEastFacingRoofSectionSum
  m2: 54

Row 7:
  direction: SouthWestFacingRoofSectionSum
  m2: 0

Row 8:
  direction: WestFacingRoofSectionSum
  m2: 0

Row 9:
  direction: AreaIndeterminableRoofSectionSum
  m2: 0

UPRN: 10067623656
- Roof material
Row 1:
  roofMaterial: http://ies.data.gov.uk/ontology/ies-building1#WaterproofMembrane

## Run EPC/Building queries for sample UPRNs

In [ ]:
for u in uprns:
    print('='*80)
    print('UPRN:', u)
    print('- Building (lodgementDate, builtForm, structureUnitType)')
    rows = run(get_building(u))
    pprint_rows(rows)

    print('- Roof (roofConstruction, roofInsulation, roofInsulationThickness)')
    rows = run(get_roof_for_building(u))
    pprint_rows(rows)

    print('- Floor (floorConstruction, floorInsulation)')
    rows = run(get_floor_for_building(u))
    pprint_rows(rows)

    print('- Walls & Windows (wallConstruction, wallInsulation, windowGlazing)')
    rows = run(get_walls_and_windows_for_building(u))
    pprint_rows(rows)


UPRN: 10003320518
- Building (lodgementDate, builtForm, structureUnitType)
Row 1:
  lodgementDate: http://iso8601.iso.org/2024-10-31
  builtForm: http://ies.data.gov.uk/ontology/ies-building1#SemiDetached
  structureUnitType: http://ies.data.gov.uk/ontology/ies-building1#Bungalow

- Roof (roofConstruction, roofInsulation, roofInsulationThickness)
Row 1:
  uprn: None
  roofConstruction: http://ies.data.gov.uk/ontology/ies-building1#PitchedRoof
  roofInsulation: http://ies.data.gov.uk/ontology/ies-building1#LoftInsulation
  roofInsulationThickness: http://ies.data.gov.uk/ontology/ies-building1#100mm_Insulation

- Floor (floorConstruction, floorInsulation)
Row 1:
  uprn: None
  floorConstruction: http://ies.data.gov.uk/ontology/ies-building1#Suspended
  floorInsulation: http://ies.data.gov.uk/ontology/ies-building1#NoInsulationInFloor

- Walls & Windows (wallConstruction, wallInsulation, windowGlazing)
Row 1:
  uprn: None
  wallConstruction: http://ies.data.gov.uk/ontology/ies-building1#C

## Notes
- This notebook simulates a Fuseki endpoint by running queries over a single in-memory graph that contains all triples (ontology + data).
- If you need to test additional iris-api queries (e.g., EPC-derived ones), add them similarly and point at the same graph.
- The `ies-building.ttl` ontology is loaded to ensure class and property IRIs resolve correctly for reasoning-free SPARQL matching.